# Data validation: does the ball-by-ball file say what it claims to say?

Before any analysis, this notebook checks the provided dataset for internal consistency and records every convention the file follows. It ends by writing `data/dpl_prelims_clean.parquet`, the file every analysis notebook reads. Nothing here removes rows; every fix is either a repair of a documented placeholder value or a new column that sits beside the raw one.

**How to read this notebook.** Each section states what is being checked and why it matters, runs the check (the code lives in `scripts/data_validation/`), and then records what was found. A check "passes" when the data agree with the rule; a "fail" is a lead to follow, not necessarily an error, and each one below is either explained or repaired. The word *innings* means one team's turn to bat; a T20 match has two. A *legal delivery* is a ball that counts toward the over (not a wide or a no-ball). *Extras* are runs not scored off the bat: wides, no-balls, byes and leg byes.

**Order of work.** Load and validate the schema, then check granularity and ordering, the run and ball ledgers, dismissals, the chase columns, names and ids, missing values, the shot-placement geometry, the two model columns (win probability and projected score), and the manually tagged columns. The cleaning step applies every repair and derived column established along the way, and writes the parquet.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("scripts").resolve()))

from common.config import load_settings
from common import results_io
from data_validation import load_raw
from data_validation import ledger_checks as ledger
from data_validation import dismissal_checks as dismissals
from data_validation import chase_checks as chase
from data_validation import entity_checks as entities
from data_validation import missingness_checks as missingness
from data_validation import wagon_wheel_checks as wagon
from data_validation import win_probability_checks as winprob
from data_validation import tagging_checks as tagging
from data_validation import build_clean_dataset as cleaning

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

settings = load_settings()
ANALYSIS = "data_validation"
SEED = settings.seed("global")
PRODUCER = "data_validation.ipynb"
RAW_CSV = settings.data_path("raw_csv")
VALIDATED = settings.data_path("validated_dataset")
CLEANED = settings.data_path("cleaned_dataset")
print(f"raw={RAW_CSV.name}  validated={VALIDATED.name}  cleaned={CLEANED.name}")

raw=Data_Premier_League_Prelims_Dataset.csv  validated=dpl_prelims.parquet  cleaned=dpl_prelims_clean.parquet


## Load and validate the schema

The CSV is read once and checked column by column against a declared schema (`scripts/data_validation/cricket_schema.py`): the type each column must have, whether it may be blank, and which values are legal. All violations are collected together rather than stopping at the first. The validated frame is written to parquet and reused while it is newer than both the CSV and the schema file. Two enrichments happen at this stage: `match_date` becomes a real date, and the fixed-label columns (teams, ground, handedness, bowler kind, outcome) become categoricals.

In [2]:
df = load_raw.load_raw_dataset(RAW_CSV, VALIDATED)
print(f"rows={len(df):,}  matches={df['p_match'].nunique()}  innings={df.groupby(['p_match', 'inns']).ngroups}")
print(f"seasons: {sorted(df['year'].unique())}   dates {df['match_date_dt'].min().date()} to {df['match_date_dt'].max().date()}")
df.head()

rows=34,967  matches=145  innings=289
seasons: [np.int64(2023), np.int64(2024)]   dates 2023-03-31 to 2024-05-26


,row_id,p_match,inns,bat,p_bat,team_bat,bowl,p_bowl,team_bowl,ball,ball_id,outcome,score,out,dismissal,p_out,over_num,noball,wide,byes,...,bat_hand,bowl_style,bowl_kind,batruns,ballfaced,bowlruns,bat_out,wagonx,wagony,wagonzone,line,length,shot,control,predscore,wprob,bowl_speed_category,bowl_release_point,bowl_type,match_date_dt
0,225793,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,1,0.01,no run,0,False,NaN,379140,1,0,0,0,...,LHB,RF,pace bowler,0,1,0,True,130.0,191.0,6.0,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,162.0,49.42,3.0,2.0,OUTSWINGER,2023-03-31
1,225794,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,2,0.02,leg bye,1,False,NaN,379140,1,0,0,0,...,LHB,RF,pace bowler,0,1,0,True,0.0,0.0,0.0,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,FLICK,0.0,161.0,48.27,3.0,2.0,OUTSWINGER,2023-03-31
2,225795,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,3,0.03,no run,0,False,NaN,1060380,1,0,0,0,...,RHB,RF,pace bowler,0,1,0,True,159.0,239.0,5.0,OUTSIDE_OFFSTUMP,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,160.0,47.12,2.0,2.0,ANGLED IN,2023-03-31
3,225796,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,4,0.04,run,1,False,NaN,1060380,1,0,0,0,...,RHB,RF,pace bowler,1,1,1,True,211.0,252.0,4.0,ON_THE_STUMPS,GOOD_LENGTH,ON_DRIVE,1.0,160.0,47.12,2.0,2.0,OUTSWINGER,2023-03-31
4,225797,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,5,0.05,no run,0,False,NaN,379140,1,0,0,0,...,LHB,RF,pace bowler,0,1,0,True,172.0,198.0,5.0,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,158.0,44.82,3.0,2.0,INSWINGER,2023-03-31


## Granularity and ordering

One row should be one physical delivery, whether or not it was legal. Three things pin that down: the (match, innings, ball id) key is unique, `row_id` is unique, and `ball_id` decodes exactly from the over number and the delivery count within the over. Then the running totals (innings runs and balls, the striker's runs and balls faced) must never fall when rows are read in delivery order; a drop would mean the rows are out of sequence or a value was mistyped.

In [3]:
granularity = ledger.check_granularity(df)
monotonic_violations = ledger.report_monotonic_ordering(df)

delivery key unique      : PASS
row_id unique            : PASS
ball_id decodes          : PASS
PASS: all cumulative columns non-decreasing in delivery order.


Found: one row per delivery, `row_id` is a primary key, `ball_id` decodes exactly, and every cumulative column is non-decreasing in delivery order. The rows can be trusted as a sequence.

## Runs per delivery

`score` should be the runs off the bat plus extras. The identity is conditional on the no-ball flag: on a legal ball byes and leg byes are part of the score; on a no-ball the score carries only the one-run penalty, and any byes on that ball are recorded separately. The check tests that rule ball by ball.

In [4]:
runs_identity_violations = ledger.check_delivery_runs_identity(df)

PASS: conditional runs identity holds on every ball.


Found: the conditional identity holds on every ball. `score` is the batting team's total for the delivery and can be summed for run rates.

## Scoring convention for the bowler

Under the laws of cricket a bowler is charged the runs off the bat, wides, and the no-ball penalty, but never byes or leg byes. The feed's `bowlruns` column instead nets byes off the bowler on a no-ball, so the fourteen balls that were both a no-ball and a bye carry a negative or zero charge. The first check confirms the feed's own convention holds everywhere; the second shows those fourteen balls break the standard convention. A new column `bowler_charge` is then built under the standard convention and reconciled against the bowler's running total `cur_bowl_runs`, and the innings total `inns_runs` is reconciled from the per-ball components.

In [5]:
feed_convention_violations = ledger.check_bowler_runs_identity(df, normalise_byes_off_noball=False)
standard_convention_violations = ledger.check_bowler_runs_identity(df, normalise_byes_off_noball=True)
cleaned = ledger.add_bowler_charge(df)
bowler_reconciled, bowler_gaps = ledger.reconcile_bowler_charge(cleaned)
innings_reconciled, innings_gaps, first_innings_gap = ledger.reconcile_innings_total(cleaned)

Stacked no-ball+bye balls: 14
PASS: bowler-runs identity holds (given the current convention).
Stacked no-ball+bye balls: 14
FAIL: 14 row(s) break the identity.
diff
-4    6
-1    8
 row_id  p_match  inns  ball_id outcome  noball  wide  byes  legbyes  batruns  bowlruns  expected_bowlruns  diff
 227242  1359480     2    19.02 leg bye       1     0     0        1        0         0                  1    -1
 227605  1359482     1    18.04     bye       1     0     1        0        0         0                  1    -1
 230355  1359493     2     9.04 leg bye       1     0     0        1        0         0                  1    -1
 230737  1359495     1    10.05 leg bye       1     0     0        4        0        -3                  1    -4
 234943  1359512     1    15.05 leg bye       1     0     0        1        0         0                  1    -1
 238814  1359528     2    12.03 leg bye       1     0     0        4        0        -3                  1    -4
 240966  1359537     2    1

Found: the feed's `bowlruns` uses the non-standard netting on exactly the stacked no-ball-plus-bye balls; the running total `cur_bowl_runs` uses the standard convention, so `bowler_charge` reconciles with it exactly for every bowler, and `inns_runs` reconciles with the per-ball components for every innings. Economy and any bowling rate must use `bowler_charge`, never `bowlruns`.

## Balls ledger and the allowance repair

Four rules: the innings ball count steps by exactly one on legal deliveries and zero otherwise; balls bowled plus balls remaining equals the innings allowance `max_balls` on every ball; every over has six legal balls except a short final over; and `ball_id` decodes to the legal-ball sequence. Failures are then diagnosed rather than counted: an allowance of 0 is a placeholder for a missing value, and the true allowance can be read back from balls bowled plus balls remaining.

In [6]:
ledger_checks, ledger_frame = ledger.check_balls_ledger(cleaned)
balls_remaining_bad, balls_remaining_summary = ledger.diagnose_balls_ledger(cleaned, ledger_checks)
ledger.implied_allowance_report(cleaned, sorted(balls_remaining_bad["p_match"].unique()))
cleaned, repaired_rows, repaired_innings = ledger.repair_max_balls(cleaned)
step_reconciliation = ledger.reconcile_step_against_over_integrity(cleaned, ledger_checks)

inns_balls step          : FAIL (2)
balls-remaining identity : FAIL (245)
over integrity           : PASS
ball_id decode           : PASS  [convention: ball_id_over == over-1; direct match 0.0%, offset match 100.0%]
=== inns_balls step failures (with surrounding context) ===
 row_id  p_match  inns  ball_id  over_num  ball  noball  wide  inns_balls outcome
 251305  1426270     1     5.06         6     6       0     0          36  no run
 251306  1426270     1     6.01         7     1       0     1          36    wide
 251307  1426270     1     6.02         7     2       1     0          37 no ball
 251308  1426270     1     6.03         7     3       0     0          37    four
  -> inns for this match: [np.int64(1), np.int64(2)]
------------------------------------------------------------
 row_id  p_match  inns  ball_id  over_num  ball  noball  wide  inns_balls outcome
 251306  1426270     1     6.01         7     1       0     1          36    wide
 251307  1426270     1     6.02     

Found: two innings carried `max_balls = 0`; both were complete 120-ball innings, and the allowance is repaired in place from balls bowled plus balls remaining. Two step failures sit in one over where the ball counter ticked one delivery late; that over has the correct six legal balls, so the per-over structure is authoritative and the step flags are timing artefacts. Any analysis that needs the allowance (rain-reduced innings, balls remaining) reads the repaired `max_balls`; the innings minimum is the binding value because a reduction can arrive mid-innings.

## Dismissals

The out flag, the dismissal method, the dismissed batter's id, the innings wicket count and the bowler's wicket tally should agree. Two conventions matter: a batter who retires hurt is flagged `out = True` but is not a dismissal, so wicket counts exclude that case; and `p_out`, which the data dictionary says is blank when no wicket fell, is in fact populated on every ball with the striker's id. A new `p_out_clean` restores the dictionary definition and is reconciled three independent ways.

In [7]:
dismissal_checks, dismissal_frame = dismissals.check_wickets_dismissals(cleaned)
print(cleaned["dismissal"].value_counts(dropna=False).head(20))
dismissals.report_p_out_convention(cleaned)
cleaned = dismissals.add_p_out_clean(cleaned)
p_out_mismatches = dismissals.reconcile_p_out_clean(cleaned)

out<->dismissal coherence        : FAIL (33168)
striker p_out mismatch           : PASS
unexpected non-striker dismissal : PASS
inns_wkts reconciliation         : PASS
inns_wkts > 10                   : PASS
bowler-wkt credit                : PASS
unknown dismissal terms          : PASS
dismissal
NaN                       33168
caught                     1270
bowled                      268
leg before wicket           115
run out                     107
stumped                      33
retired not out (hurt)        2
retired out                   2
hit wicket                    1
obstructing the field         1
Name: count, dtype: int64
0 true NaN in p_out; 0 empty-string in p_out
Non-wicket balls where p_out != p_bat: 0
p_out_clean populated on 1797 balls (should equal total genuine dismissals).
PASS: p_out_clean count matches final inns_wkts for all 289 innings.
p_out_clean populated        : 1797
out flag (excl. retired-hurt): 1797
sum of final inns_wkts       : 1797
PASS: all three 

Found: the only coherence failure is `p_out` being populated on every ball, which is the feed's convention rather than an error (the count of failing rows equals the count of non-wicket balls). Striker attribution, non-striker dismissals, the wicket count and the bowler's credited wickets all reconcile, and the dismissal vocabulary contains nothing unknown. Wickets are read from `out` or `dismissal`; `p_out_clean` identifies the dismissed batter; raw `p_out` must never be used as a wicket indicator.

## Chase columns

In the second innings the target, runs remaining and required rate must be present and consistent: the target is the first-innings total plus one, runs remaining is target minus runs scored (floored at zero when the winning hit overshoots), the run rates recompute from the counts, and the chase succeeded exactly when the chasing team is the recorded winner. Failures are diagnosed against the two legitimate causes, a rain-revised target and an abandoned match.

In [8]:
chase_checks = chase.check_chase_consistency(cleaned)
chase.diagnose_chase_consistency(cleaned, chase_checks)
for p_match in chase_checks["target_link"].index:
    target_counts, winner_counts = chase.match_target_and_winner(cleaned, p_match)
    print(f"match {p_match}: target values {target_counts.to_dict()}  winner values {winner_counts.to_dict()}")

chase_cols_inns2_missing    : PASS
chase_cols_inns1_present    : PASS
target_link                 : FAIL (3)
runs_rem_identity           : PASS
rr_mismatch                 : PASS
rrr_mismatch                : PASS
winner_consistency          : FAIL (2)
=== target-link failures ===
  match 1359519: first_inns_total=125, target=nan, expected=126, max_balls=120 -> INSPECT
  match 1370353: first_inns_total=214, target=171, expected=215, max_balls=90 -> DLS/reduced (target revised)
  match 1426298: first_inns_total=157, target=159, expected=158, max_balls=96 -> DLS/reduced (target revised)

=== winner-consistency failures ===
                        chasing_team  chase_runs  target                       winner  chase_succeeded  winner_is_chaser  super
p_match                                                                                                                        
1359479  Royal Challengers Bengaluru         172   172.0  Royal Challengers Bangalore             True             

Found: the chase columns are populated only in the second innings and the identities hold on every ball. Three target-link failures are two rain-revised targets (reduced allowance) and one abandoned match with no target and a "-" winner. The two winner-consistency failures are the same team recorded under two spellings, handled in the next section. `target`, `inns_runs_rem` and `inns_rrr` can be used as chase state; a first innings has them blank.

## Names and ids

Player names should map to one id and ids to one name, a player's batting hand and bowling style should be the same on every ball, and the winner and toss fields should name one of the match's two teams. Royal Challengers Bangalore became Bengaluru in 2024; the feed uses the new name in the team columns and the old one in winner and toss, which is the only referential failure. The canonicalisation prints counts before and after so it is auditable.

In [9]:
entity_checks = entities.check_entity_integrity(cleaned)
print(entity_checks["referential"].to_string(index=False))
cleaned = entities.canonicalise_team_names(cleaned)

bat_multi_id                      : PASS
p_bat_multi_name                  : PASS
bowl_multi_id                     : PASS
p_bowl_multi_name                 : PASS
bat_hand_unstable                 : PASS
bowl_style_unstable               : PASS
bowl_kind_unstable                : PASS
referential                       : FAIL (16)
 p_match  field                       value                                                teams
 1359479 winner Royal Challengers Bangalore        {Royal Challengers Bengaluru, Mumbai Indians}
 1359479   toss Royal Challengers Bangalore        {Royal Challengers Bengaluru, Mumbai Indians}
 1359483   toss Royal Challengers Bangalore {Kolkata Knight Riders, Royal Challengers Bengaluru}
 1359494 winner Royal Challengers Bangalore        {Delhi Capitals, Royal Challengers Bengaluru}
 1359498   toss Royal Challengers Bangalore   {Chennai Super Kings, Royal Challengers Bengaluru}
 1359501 winner Royal Challengers Bangalore          {Punjab Kings, Royal Challengers

Found: ids and names are one-to-one and player attributes are stable. One team is spelled two ways across columns; every occurrence is now the 2024 spelling, and the earlier winner-consistency failures clear with it. Ten teams, one spelling each.

## Missing values and the control column

The tracked and tagged columns (line, length, shot, control, placement coordinates, speed, release point, bowl type) are checked for how often they are blank, whether blanks cluster by match, and whether the control blank rate is stable across seasons. Control, which says whether the batter middled the ball, is meaningful only on a faced ball; the check separates faced balls with no control (a small coding gap) from unfaced balls with a control value (a leak onto wides).

In [10]:
missingness_checks = missingness.check_missingness(cleaned)
cleaned = missingness.add_control_clean(cleaned)

blank rates:
    bowl_speed_category   : 6.8%
    bowl_release_point    : 6.8%
    bowl_type             : 5.5%
    control               : 2.8%
    shot                  : 0.5%
    line                  : 0.2%
    length                : 0.1%
    wagonx                : 0.0%
    wagony                : 0.0%
    wagonzone             : 0.0%

per-match all-or-nothing test:
    line                  : 0.0% matches fully blank, 99.3% fully present
    length                : 0.0% matches fully blank, 99.3% fully present
    shot                  : 0.0% matches fully blank, 86.9% fully present
    control               : 0.0% matches fully blank, 15.2% fully present
    wagonx                : 0.0% matches fully blank, 100.0% fully present
    wagony                : 0.0% matches fully blank, 100.0% fully present
    wagonzone             : 0.0% matches fully blank, 100.0% fully present
    bowl_speed_category   : 0.0% matches fully blank, 2.8% fully present
    bowl_release_point    : 0.0

Found: blank rates are below 7% in every column, no match is entirely blank in any column, and the control blank rate is the same in both seasons (2.6% and 2.9%), so a season contrast on a tagged column is not confounded by a coverage change. Control is set on 447 wides, which are not faced balls; `control_clean` nulls those and is the column any control rate must use. The 55 faced balls with no control stay blank and are excluded from rates, never scored as controlled.

## Shot placement geometry

Each scoring shot carries wagon-wheel coordinates and a fielding zone from 1 to 8. The checks ask whether the eight zones are distinct angular sectors around a fixed origin, whether the coordinates are already batter-relative (so left and right-handers can be pooled), and what the (0, 0) coordinate means. Angles that straddle the 0/360 degree seam need circular statistics; a plain mean would place such a zone in the wrong quadrant.

In [11]:
wagon_checks = wagon.check_wagon_geometry(cleaned)
scored_without_zone = wagon.diagnose_scored_without_zone(cleaned)
left_alone_with_zone = wagon.diagnose_left_alone_with_zone(cleaned)
cleaned = wagon.backfill_wagonzone(cleaned)
wagon.sentinel_coordinate_report(cleaned)
cleaned = wagon.add_has_placement(cleaned)

Origin (range-midpoint): x=181.5, y=181.5

per-zone linear vs circular angle stats:
           count  lin_mean  lin_std  circ_mean  circ_std      R                                      verdict
wagonzone                                                                                                   
1.0         1578     290.9     10.1      290.9      10.1  0.985                                        TIGHT
2.0         2702     328.5     11.2      328.6      11.2  0.981                                        TIGHT
3.0         5904      51.4    100.2       15.9      13.1  0.974  TIGHT (linear inflated by 0/360 wraparound)
4.0         4456      66.3     32.2       62.8      19.3  0.945                                        TIGHT
5.0         4252     121.2     20.6      120.6      20.0  0.941                                        TIGHT
6.0         5748     165.6     13.3      165.5      13.2  0.974                                        TIGHT
7.0         3023     211.4     10.9      211

Found: the eight zones are tight, distinct sectors once angles are computed circularly (zone 3 straddles the seam; its linear spread was an artefact). Left and right-handed distributions are aligned, so coordinates are already normalised for handedness and pool directly. (0, 0) is a missing-coordinate placeholder: it sits on 5,628 balls, almost all dot balls, wides, extras and wickets, plus 24 scoring balls whose placement was simply not recorded. `wagonzone_clean` keeps every real zone and leaves those 24 blank rather than inventing a position; `has_placement` flags the 29,339 balls with genuine coordinates. Sixty-eight balls tagged as "left alone" carry a zone and 55 of them scored runs, a shot-tag contradiction sized here and carried as part of the tagging error floor.

## Win probability and projected score

Two model columns come with the feed: `wprob`, the batting team's win probability in percent, and `predscore`, the projected innings total. Four questions: do the extreme values (0 and 100) sit where they should, late in chases; does the last ball converge on the result; is the value stated before or after the ball (a boundary should raise `wprob` on the same row if it is stated after); and is it calibrated, meaning that among balls where it says 60%, the batting side won about 60% of the time. A final test holds the probability fixed in the 40 to 60% band and asks whether stronger teams still win more often than the model says.

In [12]:
extreme_wprob = winprob.extreme_wprob_report(cleaned)
last_ball, last_chase_ball = winprob.check_terminal_convergence(cleaned)
large_terminal_gaps = winprob.terminal_predscore_gaps(cleaned)
wprob_timing = winprob.check_wprob_timing(cleaned)

Innings phase of extreme wprob values (over_num):
min      9.0
25%     19.0
50%     20.0
mean    18.9
max     20.0

Innings distribution (should skew to 2nd innings / death overs):
inns
2    144

Wickets down at these points (should skew high for the 0s):
min      0.0
50%      6.0
mean     6.2
max     10.0
predscore vs actual final inns_runs (at last ball of innings):
    mean abs gap: 5.68, median: 0.00, max: 117
    exact matches: 227 of 289 innings
    within 5 runs: 80.6%

wprob at final ball of chase:
    chase WON  (69): mean wprob 100.0 (want ~100), min 100
    chase LOST (75): mean wprob 0.0 (want ~0), max 0
Innings with terminal predscore gap > 5: 56
 p_match  inns  over_num  inns_balls  inns_balls_rem  max_balls  inns_runs  predscore   gap  wprob
 1426295     2        10          58              62        120        167      284.0 117.0  100.0
 1426312     2        11          63              57        120        114      219.0 105.0  100.0
 1426270     2         9          5

In [13]:
reliability_all = winprob.check_wprob_calibration(cleaned)
reliability_first_innings = winprob.check_wprob_calibration(cleaned, inns_filter=1)
reliability_second_innings = winprob.check_wprob_calibration(cleaned, inns_filter=2)
reliability_death_chase = winprob.check_wprob_calibration(cleaned, inns_filter=2, min_over=15, n_bins=5)

Reliability diagram (34848 balls, decided matches):

                  n  predicted  actual    gap
bin                                          
(-0.001, 0.1]  5400      0.031   0.100  0.069
(0.1, 0.2]     2369      0.148   0.282  0.134
(0.2, 0.3]     1980      0.248   0.274  0.026
(0.3, 0.4]     2445      0.354   0.392  0.038
(0.4, 0.5]     3992      0.453   0.467  0.014
(0.5, 0.6]     4130      0.549   0.499 -0.050
(0.6, 0.7]     3398      0.650   0.567 -0.083
(0.7, 0.8]     3770      0.750   0.637 -0.114
(0.8, 0.9]     3309      0.850   0.728 -0.122
(0.9, 1.0]     4055      0.960   0.914 -0.046

Expected Calibration Error (ECE): 0.0687  (lower = better)
'actual' should track 'predicted' down the table; 'gap' near 0 in every bin.
Reliability diagram (18017 balls, decided matches, inns=1):

                  n  predicted  actual    gap
bin                                          
(-0.001, 0.1]   153      0.070   0.137  0.068
(0.1, 0.2]      404      0.156   0.255  0.099
(0.2, 0.3]   

In [14]:
team_awareness_pooled = winprob.check_wprob_team_awareness(cleaned)
team_awareness_by_year = winprob.check_wprob_team_awareness(cleaned, by_year=True)


ALL YEARS
  team strength (win rate), median=0.509:
    0.607  strong  Kolkata Knight Riders
    0.586  strong  Chennai Super Kings
    0.552  strong  Gujarat Titans
    0.552  strong  Rajasthan Royals
    0.536  strong  Lucknow Super Giants
    0.483  weak    Royal Challengers Bengaluru
    0.433  weak    Mumbai Indians
    0.433  weak    Sunrisers Hyderabad
    0.429  weak    Delhi Capitals
    0.393  weak    Punjab Kings
  dispersion: std=0.076, range=[0.39, 0.61]  (10 teams)
  mid-range chase balls (wprob 40-60, inns 2): 1740
              n  mean_wprob  actual_win
strong_bat                             
weak_bat    980       0.500       0.486
strong_bat  760       0.499       0.604
  predicted differs by strength: 0.001
  ACTUAL win rate differs by strength: 0.118
  -> TEAM-AGNOSTIC (flat predicted, actual varies)

YEAR 2023
  team strength (win rate), median=0.500:
    0.667  strong  Chennai Super Kings
    0.647  strong  Gujarat Titans
    0.571  strong  Lucknow Super Giants
  

Found: the extreme values sit late in second innings at extreme states, and the final ball of every chase reads 100 for a successful chase and 0 for a failed one. The projected score matches the final total exactly for most innings; the large gaps are innings that ended early (a chase won with balls to spare, or all out). A boundary moves `wprob` on its own row by +3.5 points and a wicket by −4.2, against about +1 and −0.6 on the following row, so the value is stated after the ball and the ball-to-ball change is a valid measure of that ball's impact. Calibration is good in chases and weakest in the first innings, where the model is over-confident in the 60 to 90% range. Holding the probability at 40 to 60%, the model's prediction is flat across team strength while the actual win rate differs by about 12 points, so the model is team-agnostic: raw win-probability impact under-credits players on strong teams. None of the analyses in this project use `wprob` or `predscore` as an input; they are noted here so that any later use reads them against a team baseline.

## Manually tagged columns

`bowl_type`, `length`, `line` and `shot` are tagged by hand. The audit lists each column's vocabulary, looks for terms that leak between columns, flags physically contradictory pairs on one ball (a bouncer tagged as full, a yorker tagged as short), cross-tabulates bowl type against length so unexpected combinations can be seen, and checks where the bowl-type tag is missing. Finally the 31 bowl-type labels are collapsed into nine families for any analysis that needs a coarser grouping.

In [15]:
tag_audit = tagging.audit_tag_consistency(cleaned)
type_length_contradictions = tagging.flag_type_length_contradictions(cleaned)
bowl_type_missing = tagging.bowl_type_missing_by_phase(cleaned)
cleaned = tagging.collapse_bowl_type(cleaned)

=== Value vocabularies (inspect for cross-column leakage) ===

bowl_type (31 distinct):
bowl_type
ANGLED IN              5260
STRAIGHT BALL          4184
ANGLED ACROSS          3172
LEG SPIN               2941
KNUCKLEBALL            2698
ORTHODOX               2293
OFF SPIN               1962
NaN                    1936
OFF CUTTER             1669
GOOGLY                 1411
OUTSWINGER             1134
SLOWER BALL             953
ARM BALL                931
INSWINGER               923
BOUNCER                 706
CARROM BALL             676
CHINAMAN                541
WRONG ONE               327
LEGCUTTER               280
SLOW BOUNCER            274
BACKHAND SLOWER         222
NIPBACKER               122
FLIPPER                 105
NIPPED AWAY              96
SLIDER                   66
FASTER ONE               45
REVERSE SWING            14
TOP SPINNER              13
SLOW YORKER               5
REVERSESWING YORKER       4
DRIFTER                   3
INSWINGING YORKER         1

lengt

Found: the vocabularies are disjoint and no term is misfiled. Length and bowl type are orthogonal axes (where the ball pitched versus how it was delivered), so only the yorker and bouncer families can contradict the length column; 157 balls (0.45%) carry a bouncer tag with a full length. The bowl-type tag is missing three times as often in the death overs (11%) as in the powerplay or middle (4%) in both seasons, so that missingness is informative and the untagged balls are kept as their own `unclassified` group rather than dropped. Together with the control leak and the left-alone contradictions, this is the basis for treating every tagged column as carrying an error floor of several percent, checked to be the same in both seasons.

## The cleaning step and the written file

Every repair and derived column above is applied once more by `apply_cleaning`, in the order the checks established them, to the validated frame; the result is written to `data/dpl_prelims_clean.parquet`. The table below lists the added columns. The frame built step by step in this notebook and the frame from `apply_cleaning` must be identical; the assertion checks that.

In [16]:
clean_dataset = cleaning.apply_cleaning(df)
for col in cleaning.ADDED_COLUMNS + ["max_balls", "winner", "toss"]:
    a, b = clean_dataset[col], cleaned[col]
    same = (a.astype(str) == b.astype(str)).all() if a.dtype == object or str(a.dtype) == "str" else ((a == b) | (a.isna() & b.isna())).all()
    assert bool(same), f"{col} differs between the step-by-step frame and apply_cleaning"
path = cleaning.write_clean_dataset(clean_dataset, CLEANED)
summary = cleaning.added_columns_summary(clean_dataset)
results_io.write_table(summary, settings, ANALYSIS, "added_columns", SEED, PRODUCER,
                       {"rows": int(len(clean_dataset)), "matches": int(clean_dataset["p_match"].nunique()),
                        "written": str(path)})
print(f"written {path}  rows={len(clean_dataset):,}  columns={clean_dataset.shape[1]}")
summary

Repaired 245 rows; 2 innings.
p_out_clean populated on 1797 balls (should equal total genuine dismissals).
Replaced 'Royal Challengers Bangalore' -> 'Royal Challengers Bengaluru' per column:
  team_bat  : 0 -> 0 remaining
  team_bowl : 0 -> 0 remaining
  winner    : 1646 -> 0 remaining
  toss      : 1875 -> 0 remaining
Total 'Royal Challengers Bangalore' occurrences left across team columns: 0
control non-null: 34004 -> 33557 (removed 447 not-faced)
Backfilled 0 zone(s) from real coordinates; 24 hit-but-no-usable-coord (incl. (0,0) sentinel) left as NaN.
Balls with real wagon placement: 29339 of 34967
bowl_type_group
seam_stock               12616
wrist_spin                6067
finger_spin               5186
slower_ball_deception     3918
swing                     2289
cutter                    1949
unclassified              1936
bouncer                    980
other_rare                  16
yorker                      10
written /Users/madkad/dpl_2026_s1/data/dpl_prelims_clean.parquet 

,column,dtype,non_null,distinct
0,bowler_charge,int64,34967,8
1,p_out_clean,float64,1797,202
2,control_clean,float64,33557,2
3,wagonzone_clean,float64,29339,8
4,has_placement,bool,34967,2
5,bowl_type_group,str,34967,10


**Conventions carried into every analysis.** Run rate uses `score` over legal balls; striker rates (boundary, six, dot) use `batruns` over faced balls (`ballfaced == 1`); economy uses `bowler_charge`; wickets come from `out` or `dismissal`; control comes from `control_clean`; the allowance is the innings minimum of the repaired `max_balls`; chase state is blank in the first innings; a (0, 0) placement is missing, not a position; team names have one spelling.